In [2]:
import json, base64, re
import pickle as pkl

from typing import TypedDict, Dict, Any, List

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

from langchain.globals import set_debug, set_verbose
set_debug(False)
set_verbose(False)

In [4]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

In [5]:
import base64

def encode_image(image_path):
    """Encode a local image to base64 string."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')
    
def invoke_llm(model, SYSTEM_PROMPT, user_message):

    messages =[
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_message)
    ]

    response = model.invoke(messages)
    return response.text()

In [23]:
OBSERVER_PROMPT = """
Role: Dermatologist specializing in dermoscopy.

Task: Analyze the dermoscopic image and produce a complete, objective, structured JSON of all specified dermoscopic features, organized by visual category.

How to Think: Scan the image thoroughly by this order center → mid → periphery order, do not skip any features you seen.

STRICT RULES:
- Your output must be ONLY the JSON object specified in the schema. No prose.
- You must populate every single key in the schema. Do not omit any feature.
- Use enums only: state must be "present", "absent", or "unknown". 
- If you cannot determine the state of a feature, set state="unknown"

REUSABLE SUB-SCHEMA:
    FeatureField = { "state":"present|absent|unknown", detail": "Short details" }

JSON Schema:
{
  "global": {
    "symmetry": { "shape": "symmetric|asymmetric|unknown", "color": "symmetric|asymmetric|unknown" },
    "pattern": "reticular|globular|homogeneous|multicomponent|structureless|unknown"
  },
  "border": {
    "demarcation": "well|ill|partial|unknown",
    "shape": "regular|irregular|notched|fading|unknown"
  },
  "colors": {list of colors present},
    "structures": {
      "network": {
        "typical": FeatureField,
        "atypical": FeatureField,
        "pseudonetwork": FeatureField
      },
    "negative_network": FeatureField,
    "dots_globules": FeatureField,
    "streaks": FeatureField,
    "blotches": FeatureField,
    "veil": FeatureField,
    "regression": FeatureField,
    "shiny_white_lines": FeatureField,
    "leaf_structures": FeatureField,
    "spoke_wheels": FeatureField,
    "ovoid_nests": FeatureField,
    "milia_cysts": FeatureField,
    "comedo_openings": FeatureField,
    "cerebriform": FeatureField,
    "central_scar": FeatureField,
    "white_network": FeatureField,                
    "peripheral_delicate_network": FeatureField,  
    "peripheral_pigment_rim": FeatureField,
    "peripheral_network": FeatureField,
    "keratin_scale": FeatureField,
    "rosettes": FeatureField,
    "white_halos": FeatureField,
    "ulcer_crust": FeatureField
  },
  "vascular": {
    "arborizing": FeatureField,
    "glomerular": FeatureField,
    "polymorphous": FeatureField,
    "dotted": FeatureField,
    "linear": FeatureField,
    "lacunae": FeatureField,
    "red_patches": FeatureField,
    "dotted_in_lines": FeatureField 
  }
}
"""

In [33]:
image_path = "../dataset/train/ISIC_0028872.jpg"
image_data = encode_image(image_path)

patient_data = {
    "age": 50,
    "sex": "male",
    "lesion_location": "posterior torso"
}

user_message = [
    {
        "type": "image_url",
        "image_url": { "url": f"data:image/jpeg;base64,{image_data}" }
    },
    {
        "type": "text",
        "text": f"Patient Data: {patient_data}"
    }
]

observer_llm  = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.0,
    top_p=0.95
)

image_features = invoke_llm(observer_llm, OBSERVER_PROMPT, user_message)
print(image_features)

```json
{
  "global": {
    "symmetry": {
      "shape": "asymmetric",
      "color": "asymmetric"
    },
    "pattern": "multicomponent"
  },
  "border": {
    "demarcation": "partial",
    "shape": "irregular"
  },
  "colors": [
    "light brown",
    "dark brown",
    "pink",
    "red",
    "white",
    "grey"
  ],
  "structures": {
    "network": {
      "typical": {
        "state": "absent",
        "detail": ""
      },
      "atypical": {
        "state": "absent",
        "detail": ""
      },
      "pseudonetwork": {
        "state": "absent",
        "detail": ""
      }
    },
    "negative_network": {
      "state": "absent",
      "detail": ""
    },
    "dots_globules": {
      "state": "present",
      "detail": "Scattered brown and red dots/globules of varying sizes"
    },
    "streaks": {
      "state": "absent",
      "detail": ""
    },
    "blotches": {
      "state": "present",
      "detail": "Ill-defined darker brown areas"
    },
    "veil": {
      "state": "

In [34]:
TUMOR_FAMILY_PROMPT = """
You are a dermatologist analyzing a lesion based on the detailed report provided by a lab technician. The report includes information about the lesion's visual features and dermatoscopic findings, additionally patient data may be provided.

Your task is to classify the lesion into one of the following tumor families based on the lesion features described in the report:

Tumor Families:
1. Melanocytic Tumor Family (e.g., melanoma, nevus)
2. Keratotic Tumor Family (e.g., seborrheic keratosis (PBK), actinic keratosis (AK), squamous cell carcinoma(SCC))
3. Basal Cell Carcinoma Tumor Family (e.g., basal cell carcinoma (BCC))
4. Dermatofibroma (e.g., dermatofibroma)

Task:
- Classify the lesion into one of the tumor families based on the provided report.
- Provide a brief reasoning for your classification, especially if the lesion has overlapping features or mimics other conditions.

If the features of the lesion make it unclear which tumor family it belongs to, note the conflict and explain your reasoning.

Expected Output (JSON Schema):
{
  "tumor_family": "string",  
  "reasoning": "string"  
}
"""

In [35]:
user_message = [
    {
        "type": "text",
        "text": f"Image Features:\n{json.dumps(image_features, ensure_ascii=False)}"
    },
    {
        "type": "text",
        "text": f"Patient Data: {patient_data}"
    }
] 

triage_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    top_p=0.95
)

triage_result = invoke_llm(triage_llm, TUMOR_FAMILY_PROMPT, user_message)
print(triage_result)

```json
{
  "tumor_family": "Melanocytic Tumor Family",
  "reasoning": "The lesion exhibits multiple highly concerning features for melanoma, including marked asymmetry in both shape and color, an irregular border, and a multicomponent pattern. The presence of six different colors (light brown, dark brown, pink, red, white, grey) is a strong indicator for melanoma. Dermatoscopic structures such as scattered brown dots/globules, ill-defined darker brown blotches, and significant areas of whitish/greyish regression are classic signs of melanocytic malignancy. Furthermore, the vascular pattern is polymorphous (mix of dotted and fine linear vessels), which is highly suspicious for melanoma. Small erosions/crusts (ulceration) can also be seen in advanced melanomas. While some features like polymorphous vessels and ulceration can be seen in aggressive Keratotic tumors (e.g., SCC), the prominent brown pigment, blotches, and regression areas are much more characteristic of a melanocytic lesion

In [36]:
DIAGNOSIS_PROMPT = """
You are a dermatologist analyzing a lesion based on the tumor family classification provided by the first dermatologist and the detailed report from the lab technician. Your task is to make a final diagnosis based on the tumor family and features described in the report. If the lesion has conflicting features or appears to mimic multiple conditions, please note the conflict and resolve the issue by providing a clear explanation.

Review the Tumor Family Classification:
	- Review the tumor family classification from the first dermatologist.
	- If you agree with the classification, proceed to confirm the diagnosis.
	- If you disagree with the classification or think the lesion mimics multiple families, provide a detailed explanation of why the lesion doesn't fit the initial classification. For example, if melanoma is suspected but there are several features of seborrheic keratosis (PBK), note this and explain why you might lean toward a different diagnosis.

Next Steps:
	- If a malignant lesion is suspected (e.g., melanoma, BCC, SCC):
		- Recommend a biopsy (e.g., excisional biopsy or punch biopsy).
		- Example recommendations:
		- “Biopsy recommended for suspected melanoma.”
		- “Excisional biopsy advised for suspected BCC.”
	- If a benign lesion is suspected (e.g., seborrheic keratosis, dermatofibroma):
		- Suggest monitoring for changes or removal if the lesion is symptomatic or cosmetically concerning.
		- Example recommendations:
		- “Monitor for changes over the next 3-6 months.”
		- “Remove if symptomatic or cosmetically concerning.”
	- If there is uncertainty or a conflict between diagnoses (e.g., features of both melanoma and seborrheic keratosis (PBK)):
		- Suggest follow-up for monitoring or digital dermoscopy to track changes.
		- Example recommendations:
		- “Follow-up in 3-6 months for monitoring.”
	- “If the lesion increases in size or develops new symptoms, consider biopsy.”

Provide your reasoning for the final diagnosis and the proposed next steps based on the lesion's features and the tumor family classification. If there's a conflict or uncertainty, explain the reasoning behind your decision and how you resolved the issue.
Expected Output (JSON Schema):
{
  "final_diagnosis": {
    "diagnosis": "string",  // e.g., "melanoma"
    "reasoning": "string"  // e.g., "The lesion's irregular borders, multiple colors (brown, black, blue), and irregular pigment network are highly suspicious for melanoma."
  },
  "next_steps": {
    "recommendation": "string",  // e.g., "excisional biopsy"
    "justification": "string"  // e.g., "An excisional biopsy is necessary to confirm the diagnosis of melanoma. Immediate biopsy is critical for accurate staging and treatment planning."
  }
}
"""

In [37]:
user_message = [
    {
        "type": "text",
        "text": f"Image Features:\n{json.dumps(image_features, ensure_ascii=False)}"
    },
    {
        "type": "text",
        "text": f"Patient Data: {patient_data}"
    }
] 

dx_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
    top_p=0.95
)

dx_result = invoke_llm(dx_llm, DIAGNOSIS_PROMPT, user_message)
print(dx_result)

```json
{
  "final_diagnosis": {
    "diagnosis": "Melanoma (highly suspected)",
    "reasoning": "The lesion exhibits marked asymmetry in both shape and color, coupled with irregular and partially demarcated borders. The presence of six distinct colors (light brown, dark brown, pink, red, white, grey), particularly the white and grey areas indicating regression, is highly concerning. Structural analysis reveals ill-defined darker brown blotches, scattered brown and red dots/globules of varying sizes, and small erosions/crusts. A polymorphous vascular pattern, characterized by a mix of dotted and fine linear vessels, further supports the suspicion of malignancy. These combined features—asymmetry, border irregularity, color variegation, regression, atypical vascularity, and ulceration—are strong dermoscopic indicators for melanoma."
  },
  "next_steps": {
    "recommendation": "Excisional biopsy",
    "justification": "Due to the high suspicion for melanoma based on the multiple concern